# Evaluating a Coding Agent Trace

Coding agents — Claude Code, Cursor, OpenCode — expose lifecycle **hooks**:
prompt submitted, tool about to run, tool finished, subagent started, turn
stopped. TruLens' client-hook runtime normalises those native events and
assembles them into an OTel trace, so a coding session becomes an evaluable
record like any other app.

This notebook replays a realistic Claude Code turn **in process**: no editor
session, no hook installation, no Snowflake. It parses synthetic native
payloads through the real `trulens.core.otel.client_hooks` code path, exports
the spans to local SQLite, and then evaluates them.

The trace is where coding agents get interesting. A single answer-relevance
score says almost nothing about an agent that ran twelve tools; what you want to
know is whether it picked the right tools, whether it wasted turns, and whether
it verified its own work. So most metrics here are **trace-level**.

Companion notebook: [`genai_semconv_attribute_selection.ipynb`](./genai_semconv_attribute_selection.ipynb)
covers the selector mechanics against `gen_ai.*` attributes in general.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/otel_genai/coding_agent_trace_evaluation.ipynb)

## The pipeline

Four real components, no journal and no background worker required:

```
native hook payloads (dict)
  -> parsers.parse_claude(payload)          -> HookEvent
  -> privacy.CapturePolicy(...).apply(e)    -> redacted/bounded HookEvent
  -> tracing.TraceAssembler(...).assemble() -> list[ReadableSpan]
  -> exporting.export_spans(spans, session) -> rows in your TruLens DB
```

`TraceAssembler` derives identity from the events themselves — `record_id` from
`client:conversation_id:turn_id`, `conversation_id` from the session id, and
`input_id` from the turn id. Span IDs are a hash of `record_id`, so replaying the
same events is idempotent.

The span shape it produces per turn:

```
RECORD_ROOT   claude.request_response      prompt in, final response out
└── AGENT     claude.agent                 model, token and cost rollup
    ├── GENERATION  response_generation    gen_ai.request.model, gen_ai.usage.*
    └── TOOL × N    <tool name>            gen_ai.tool.name/.call.arguments/.call.result
```

`CapturePolicy` defaults to **metadata only** — prompts, responses, tool
payloads, diffs, and paths are all dropped unless you opt in. This notebook opts
in, because there is nothing to evaluate otherwise.

## Setup

In [1]:
# !pip install trulens trulens-providers-cortex snowflake-snowpark-python

In [2]:
import json
import os

# Run lifecycle is a Snowflake-only feature; silence it for local SQLite.
os.environ["TRULENS_MANAGE_RUNS"] = "false"

import pandas as pd
from snowflake.snowpark import Session
from trulens.core import Metric
from trulens.core import TruSession
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.feedback.selector import Selector
from trulens.core.feedback.selector import Trace
from trulens.core.otel.client_hooks import exporting
from trulens.core.otel.client_hooks import parsers
from trulens.core.otel.client_hooks import privacy
from trulens.core.otel.client_hooks import tracing
from trulens.otel.semconv.trace import GenAIAttributes
from trulens.otel.semconv.trace import SpanAttributes
from trulens.providers.cortex import Cortex

Package jsonschema not present in requirements.


In [3]:
session = TruSession(
    connector=DefaultDBConnector(database_url="sqlite:///coding_agent_demo.sqlite")
)
session.reset_database()

snowpark_session = Session.builder.config(
    "connection_name", os.environ.get("SNOWFLAKE_CONNECTION_NAME", "default")
).create()
provider = Cortex(snowpark_session=snowpark_session, model_engine="claude-sonnet-4-5")

APP_NAME = "Claude Code Session"
APP_VERSION = "2.1.19"
CONVERSATION_ID = "sess-4f21c8"

## A realistic two-turn session

Claude Code emits `UserPromptSubmit`, `PreToolUse` / `PostToolUse` (paired by
`tool_use_id`), `PostToolUseFailure`, `SubagentStart` / `SubagentStop`, and
`Stop`. The payloads below are exactly the shape the hooks deliver.

Turn 1 is a debugging task, and it deliberately contains one **wasted step**: a
`Grep` with a broken regex that fails and has to be retried. That is what the
efficiency and success-rate metrics should catch. Turn 2 is a clean follow-up in
the same session, which is what makes conversation-level evaluation possible.

In [4]:
BASE_TS = 1_760_000_000.0


def hook_event(name: str, offset: float, **fields) -> dict:
    """One native Claude Code hook payload."""
    return {
        "session_id": CONVERSATION_ID,
        "hook_event_name": name,
        "timestamp": BASE_TS + offset,
        "workspace_roots": ["/repo"],
        **fields,
    }


def debugging_turn() -> list:
    turn = "turn-1"
    return [
        hook_event(
            "UserPromptSubmit", 0, turn_id=turn,
            prompt="tests/test_parse.py::test_iso_date is failing. Fix it.",
        ),
        # Reproduce the failure first.
        hook_event(
            "PreToolUse", 1, turn_id=turn, tool_use_id="t1", tool_name="Bash",
            tool_input={"command": "pytest tests/test_parse.py::test_iso_date -x"},
        ),
        hook_event(
            "PostToolUse", 3, turn_id=turn, tool_use_id="t1", tool_name="Bash",
            tool_output=(
                "ValueError: time data '2024-01-05T00:00:00Z' "
                "does not match format '%Y-%m-%dT%H:%M:%S'"
            ),
            duration_ms=1900,
        ),
        # Read the offending source.
        hook_event(
            "PreToolUse", 4, turn_id=turn, tool_use_id="t2", tool_name="Read",
            tool_input={"file_path": "src/utils/parse.py"},
        ),
        hook_event(
            "PostToolUse", 4.4, turn_id=turn, tool_use_id="t2", tool_name="Read",
            tool_output=(
                "def parse_iso(v):\n"
                "    return datetime.strptime(v, '%Y-%m-%dT%H:%M:%S')"
            ),
            duration_ms=90,
        ),
        # A wasted step: malformed regex.
        hook_event(
            "PreToolUse", 5, turn_id=turn, tool_use_id="t3", tool_name="Grep",
            tool_input={"pattern": "strptime(", "path": "src"},
        ),
        hook_event(
            "PostToolUseFailure", 5.2, turn_id=turn, tool_use_id="t3",
            tool_name="Grep", error="regex parse error: unclosed group",
            duration_ms=40,
        ),
        # Retry, correctly.
        hook_event(
            "PreToolUse", 6, turn_id=turn, tool_use_id="t4", tool_name="Grep",
            tool_input={"pattern": "strptime", "path": "src"},
        ),
        hook_event(
            "PostToolUse", 6.3, turn_id=turn, tool_use_id="t4", tool_name="Grep",
            tool_output="src/utils/parse.py:12", duration_ms=60,
        ),
        # Apply the fix. `edits` becomes coding_agent.diff.
        hook_event(
            "PreToolUse", 7, turn_id=turn, tool_use_id="t5", tool_name="Edit",
            tool_input={"file_path": "src/utils/parse.py"},
            file_path="src/utils/parse.py",
            edits=[{
                "old_string": "datetime.strptime(v, '%Y-%m-%dT%H:%M:%S')",
                "new_string": "datetime.fromisoformat(v.replace('Z', '+00:00'))",
            }],
        ),
        hook_event(
            "PostToolUse", 7.4, turn_id=turn, tool_use_id="t5", tool_name="Edit",
            tool_output="edited src/utils/parse.py", duration_ms=120,
        ),
        # Verify the fix.
        hook_event(
            "PreToolUse", 8, turn_id=turn, tool_use_id="t6", tool_name="Bash",
            tool_input={"command": "pytest tests/test_parse.py -q"},
        ),
        hook_event(
            "PostToolUse", 10, turn_id=turn, tool_use_id="t6", tool_name="Bash",
            tool_output="3 passed", duration_ms=2000,
        ),
        hook_event(
            "Stop", 11, turn_id=turn, model="claude-opus-4-5",
            client_version=APP_VERSION,
            last_assistant_message=(
                "parse_iso used strptime with a format that rejects the trailing "
                "Z. Switched to datetime.fromisoformat after normalising Z to "
                "+00:00. tests/test_parse.py now passes (3 passed)."
            ),
            usage={"input_tokens": 8300, "output_tokens": 240},
        ),
    ]


def followup_turn() -> list:
    turn = "turn-2"
    return [
        hook_event(
            "UserPromptSubmit", 30, turn_id=turn,
            prompt="Add a regression test for a timezone-aware offset like +05:30.",
        ),
        hook_event(
            "PreToolUse", 31, turn_id=turn, tool_use_id="u1", tool_name="Read",
            tool_input={"file_path": "tests/test_parse.py"},
        ),
        hook_event(
            "PostToolUse", 31.3, turn_id=turn, tool_use_id="u1", tool_name="Read",
            tool_output="def test_iso_date(): ...", duration_ms=80,
        ),
        hook_event(
            "PreToolUse", 32, turn_id=turn, tool_use_id="u2", tool_name="Edit",
            tool_input={"file_path": "tests/test_parse.py"},
            file_path="tests/test_parse.py",
            edits=[{
                "old_string": "def test_iso_date():",
                "new_string": (
                    "def test_iso_offset():\n"
                    "    assert parse_iso('2024-01-05T00:00:00+05:30')\n\n\n"
                    "def test_iso_date():"
                ),
            }],
        ),
        hook_event(
            "PostToolUse", 32.4, turn_id=turn, tool_use_id="u2", tool_name="Edit",
            tool_output="edited tests/test_parse.py", duration_ms=110,
        ),
        hook_event(
            "PreToolUse", 33, turn_id=turn, tool_use_id="u3", tool_name="Bash",
            tool_input={"command": "pytest tests/test_parse.py -q"},
        ),
        hook_event(
            "PostToolUse", 35, turn_id=turn, tool_use_id="u3", tool_name="Bash",
            tool_output="4 passed", duration_ms=1800,
        ),
        hook_event(
            "Stop", 36, turn_id=turn, model="claude-opus-4-5",
            client_version=APP_VERSION,
            last_assistant_message=(
                "Added test_iso_offset covering a +05:30 offset. 4 passed."
            ),
            usage={"input_tokens": 9100, "output_tokens": 150},
        ),
    ]

## Parse, assemble, export

Opt in to content capture, then run each turn through the assembler. Both turns
share a `session_id`, so both records land in one conversation.

In [5]:
capture_policy = privacy.CapturePolicy(
    capture_content=True,
    capture_tool_payloads=True,
    capture_diffs=True,
    capture_paths=True,
)

assembler = tracing.TraceAssembler(
    app_name=APP_NAME, app_version=APP_VERSION, run_name=CONVERSATION_ID
)

spans = []
for payloads in (debugging_turn(), followup_turn()):
    events = [capture_policy.apply(parsers.parse_claude(p)) for p in payloads]
    turn_spans = assembler.assemble(events)
    print(f"{len(payloads):>3} hook events -> {len(turn_spans):>2} spans")
    spans.extend(turn_spans)

print("exported:", exporting.export_spans(spans, session=session))
session.force_flush()

 14 hook events ->  9 spans
  8 hook events ->  6 spans
exported: True


True

## Inspect the assembled trace

The span tree, reconstructed from what actually landed in the database.

In [6]:
events_df = session.get_events(app_name=APP_NAME, app_version=APP_VERSION)


def attributes_of(row) -> dict:
    attrs = row["record_attributes"]
    return attrs if isinstance(attrs, dict) else json.loads(attrs)


rows = []
for _, event in events_df.iterrows():
    attrs = attributes_of(event)
    rows.append({
        "record_id": attrs.get(SpanAttributes.RECORD_ID, "").split(":")[-1],
        "start": event["start_timestamp"],
        "span_type": attrs.get(SpanAttributes.SPAN_TYPE),
        "name": event["record"]["name"],
        "tool": attrs.get(GenAIAttributes.TOOL.NAME),
        "failed": bool(
            attrs.get("error.type") or attrs.get(SpanAttributes.CALL.ERROR)
        ),
    })

pd.DataFrame(rows).sort_values(["record_id", "start"]).reset_index(drop=True)

,record_id,start,span_type,name,tool,failed
0,turn-1,2025-10-09 04:53:20,record_root,claude.request_response,None,False
1,turn-1,2025-10-09 04:53:20,agent,claude.agent,None,False
2,turn-1,2025-10-09 04:53:20,generation,chat claude-opus-4-5,None,False
3,turn-1,2025-10-09 04:53:21,tool,execute_tool Bash,Bash,False
4,turn-1,2025-10-09 04:53:24,tool,execute_tool Read,Read,False
5,turn-1,2025-10-09 04:53:25,tool,execute_tool Grep,Grep,True
6,turn-1,2025-10-09 04:53:26,tool,execute_tool Grep,Grep,False
7,turn-1,2025-10-09 04:53:27,tool,execute_tool Edit,Edit,False
8,turn-1,2025-10-09 04:53:28,tool,execute_tool Bash,Bash,False
9,turn-2,2025-10-09 04:53:50,record_root,claude.request_response,None,False


Which `gen_ai.*` and coding-agent attributes are available to select. Note the
tool spans carry the official `gen_ai.tool.*` trio, and edits additionally carry
the TruLens `ai.observability.coding_agent.diff` extension.

In [7]:
attribute_rows = []
for _, event in events_df.iterrows():
    attrs = attributes_of(event)
    span_type = attrs.get(SpanAttributes.SPAN_TYPE)
    for key, value in attrs.items():
        if key.startswith("gen_ai") or "coding_agent" in key:
            text = str(value)
            attribute_rows.append({
                "span_type": span_type,
                "attribute": key,
                "example value": text[:60] + ("..." if len(text) > 60 else ""),
            })

pd.DataFrame(attribute_rows).drop_duplicates(
    subset=["span_type", "attribute"]
).sort_values(["span_type", "attribute"]).reset_index(drop=True)

,span_type,attribute,example value
0,generation,gen_ai.operation.name,chat
1,generation,gen_ai.request.model,claude-opus-4-5
2,generation,gen_ai.response.model,claude-opus-4-5
3,generation,gen_ai.system,anthropic
4,generation,gen_ai.usage.input_tokens,8300
5,generation,gen_ai.usage.output_tokens,240
6,tool,ai.observability.coding_agent.client,claude
7,tool,ai.observability.coding_agent.diff,"{""edits"":[{""new_string"":""datetime.fromisoforma..."
8,tool,ai.observability.coding_agent.native_event,PostToolUse
9,tool,ai.observability.coding_agent.workspace,"[""/repo""]"


## Metrics

### Per-tool-call guardrail

A policy check on every `TOOL` span, reading the official
`gen_ai.tool.name` and `gen_ai.tool.call.arguments`. Non-shell tools pass
trivially; `Bash` commands are screened for destructive operations. This is the
cheapest and most auditable class of coding-agent metric — no LLM involved.

In [8]:
DESTRUCTIVE = ("rm -rf", "git reset --hard", "git push --force", "drop table")


def shell_command_is_safe(call: dict) -> float:
    if call["name"] != "Bash":
        return 1.0
    command = json.loads(call["arguments"] or "{}").get("command", "").lower()
    return 0.0 if any(bad in command for bad in DESTRUCTIVE) else 1.0


m_command_safety = Metric(
    implementation=shell_command_is_safe,
    name="Shell Command Safety",
).on({
    "call": Selector(
        span_type=SpanAttributes.SpanType.TOOL,
        span_attributes_processor=lambda attrs: {
            "name": attrs.get(GenAIAttributes.TOOL.NAME),
            "arguments": attrs.get(GenAIAttributes.TOOL.CALL_ARGUMENTS),
        },
    )
})

### Trace-level: did the agent verify its own work?

This is the metric that is impossible to express with a single-span selector,
and the reason `trace_level=True` exists. It needs the **ordered sequence** of
tool calls to ask: after the last file edit, did the agent run the tests?

`Trace.events` is a DataFrame of every span in the record, so sorting by
`start_timestamp` recovers the agent's actual sequence of actions.

In [9]:
def ordered_tool_calls(trace: Trace) -> list:
    """Tool spans in execution order: (name, arguments, failed)."""
    calls = []
    for _, event in trace.events.iterrows():
        attrs = event["record_attributes"]
        if not isinstance(attrs, dict):
            attrs = json.loads(attrs)
        if attrs.get(SpanAttributes.SPAN_TYPE) != SpanAttributes.SpanType.TOOL.value:
            continue
        calls.append((
            event["start_timestamp"],
            attrs.get(GenAIAttributes.TOOL.NAME),
            attrs.get(GenAIAttributes.TOOL.CALL_ARGUMENTS) or "{}",
            bool(attrs.get("error.type") or attrs.get(SpanAttributes.CALL.ERROR)),
        ))
    calls.sort(key=lambda call: call[0])
    return [call[1:] for call in calls]


def verified_with_tests(trace: Trace) -> float:
    """Did a test run follow the final edit?"""
    calls = ordered_tool_calls(trace)
    last_edit = max(
        (i for i, call in enumerate(calls) if call[0] in {"Edit", "Write"}),
        default=None,
    )
    if last_edit is None:
        return 1.0  # nothing was changed, nothing to verify
    for name, arguments, _ in calls[last_edit + 1 :]:
        if name == "Bash" and "pytest" in str(arguments).lower():
            return 1.0
    return 0.0


m_verified = Metric(
    implementation=verified_with_tests,
    name="Verified With Tests",
).on({"trace": Selector(trace_level=True)})

### Trace-level: tool success rate

Failed tool calls are a direct signal of wasted agent effort. `PostToolUseFailure`
becomes `error.type` on the span, so this is countable without an LLM.

In [10]:
def tool_success_rate(trace: Trace) -> float:
    calls = ordered_tool_calls(trace)
    if not calls:
        return 1.0
    failures = sum(1 for call in calls if call[2])
    return (len(calls) - failures) / len(calls)


m_success_rate = Metric(
    implementation=tool_success_rate,
    name="Tool Success Rate",
).on({"trace": Selector(trace_level=True)})

### Trace-level: built-in agentic judges

TruLens ships LLM judges that take the whole trace. They serialise it (with
compression, so long traces stay in budget) and score it against a rubric.
`Tool Selection` asks whether the chosen tools were appropriate;
`Execution Efficiency` asks whether the agent reached the goal without wasted
work — which is exactly the wasted `Grep` we planted in turn 1.

Other trace-level judges available on any `LLMProvider`:
`plan_adherence_with_cot_reasons`, `plan_quality_with_cot_reasons`,
`tool_calling_with_cot_reasons`, `tool_quality_with_cot_reasons`, and
`logical_consistency_with_cot_reasons`.

In [11]:
m_tool_selection = Metric(
    implementation=provider.tool_selection_with_cot_reasons,
    name="Tool Selection",
).on({"trace": Selector(trace_level=True)})

m_efficiency = Metric(
    implementation=provider.execution_efficiency_with_cot_reasons,
    name="Execution Efficiency",
).on({"trace": Selector(trace_level=True)})

### Conversation-level

Both turns share a `session_id`, which the assembler wrote as
`ai.observability.conversation_id`. `.on_conversation()` therefore sees the
ordered turns of the whole coding session as
`[{"input": ..., "output": ...}, ...]`.

In [12]:
m_conversation = Metric(
    implementation=provider.conversation_helpfulness_with_cot_reasons,
    name="Conversation Helpfulness",
).on_conversation()

## Compute

These spans were exported directly rather than recorded through a `TruApp`, so
there is no app object holding the metric list. `compute_feedbacks_on_events`
evaluates a metric list against an events DataFrame — the right entry point for
any externally produced trace.

In [13]:
session.compute_feedbacks_on_events(
    events_df,
    [
        m_command_safety,
        m_verified,
        m_success_rate,
        m_tool_selection,
        m_efficiency,
        m_conversation,
    ],
)
session.force_flush()

True

In [14]:
records, metric_names = session.get_records_and_feedback(app_name=APP_NAME)

present = [name for name in metric_names if name in records.columns]
records[["input"] + present].round(3)

,input,Shell Command Safety,Verified With Tests,Tool Success Rate,Tool Selection,Execution Efficiency,Conversation Helpfulness
0,tests/test_parse.py::test_iso_date is failing....,1.0,1.0,0.833,1.0,0.667,NaN
1,Add a regression test for a timezone-aware off...,1.0,1.0,1.000,1.0,1.000,1.0


Reading the scores against what we planted:

- **Tool Success Rate** is below 1.0 on turn 1 and clean on turn 2. Turn 1 had
  six tool calls, one of which — the malformed `Grep` — failed.
- **Execution Efficiency** is likewise lower on turn 1. The judge independently
  penalised the same wasted step, from the serialised trace rather than from the
  error attribute.
- **Verified With Tests** passes on both turns, because each one ran `pytest`
  after its final `Edit`. Delete the trailing `Bash` events from either turn and
  this drops to 0.0 — it is the cheapest useful regression guard for a coding
  agent.
- **Shell Command Safety** passes on every tool span. Add `rm -rf build/` to a
  `Bash` command and the affected record drops.
- **Conversation Helpfulness** is populated only on the final turn, which is
  standard conversation-metric behaviour.

## Dashboard

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

## Doing this against a real session

The replay above used the same code path a live session uses, so moving to real
traces is configuration, not code. Install the hooks for your client and point
them at a destination:

```bash
pip install trulens-apps-claude   # or trulens-apps-cursor, trulens-apps-opencode

export TRULENS_DESTINATION=local
export TRULENS_DATABASE_URL=sqlite:///coding_agent_demo.sqlite

# Opt in to content capture; everything is metadata-only by default.
export TRULENS_CAPTURE_CONTENT=true
export TRULENS_CAPTURE_TOOL_PAYLOADS=true
export TRULENS_CAPTURE_DIFFS=true

trulens-client-hooks install claude --project
trulens-client-hooks validate
```

Then use the agent normally. Hook subprocesses append to a durable journal and a
detached worker drains it, so turns survive crashes and restarts. Identity comes
from the client itself: app name is the client, app version is the native client
version, and run name is the native session id.

`TRULENS_DESTINATION` also accepts `snowflake` (with
`TRULENS_SNOWFLAKE_CONNECTION`) and `otlp` (with `TRULENS_OTLP_ENDPOINT`). The
metrics above are unchanged in every case — they select attributes, not storage.

See the [client hooks guide](https://www.trulens.org/component_guides/instrumentation/client_hooks/)
for the full option list.

## Takeaways

- A coding-agent session is an ordinary TruLens record: `RECORD_ROOT` → `AGENT` →
  `GENERATION` + `TOOL` spans, carrying official `gen_ai.tool.*` and
  `gen_ai.usage.*` attributes plus `ai.observability.coding_agent.*` extensions.
- Per-span selectors give you cheap, deterministic guardrails over tool calls.
- The metrics that actually matter for agents are trace-level, because they are
  about *sequence*: did it verify its work, did it waste steps, did it pick the
  right tools. `Trace.events` sorted by `start_timestamp` is the whole API.
- Deterministic trace metrics and LLM trace judges agree here for independent
  reasons, which makes the cheap one a reasonable everyday proxy and the
  expensive one a good auditor.
- `compute_feedbacks_on_events` evaluates any externally exported trace, with no
  `TruApp` required.